# VoiceHub data preparation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadirnar/voicehub/blob/main/notebooks/data_preparation.ipynb)

Create small auditable manifests, validate them, and split by speaker or recording session. Keep consent, license, and provenance with every source record.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("voicehub") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "voicehub[training] @ git+https://github.com/kadirnar/voicehub.git@main",
    ])

## 1. Configure

In [ ]:
from pathlib import Path

TTS_MODEL_TYPE = "dia"
ASR_MODEL_TYPE = "asr_wav2vec2"
DATA_ROOT = Path("data")
OUTPUT_ROOT = DATA_ROOT / "prepared"

WRITE_MANIFESTS = False
RUN_AUDIO_VALIDATION = False
RUN_MODEL_PREPARATION = False

## 2. Add source records

Use exact transcripts and stable IDs. The short examples have two groups so leakage-safe splitting can be tested.

In [ ]:
tts_records = [
    {"id": "tts-001", "text": "First authorized recording.", "audio": "tts/session-a/001.wav", "speaker_id": "speaker-1", "session_id": "session-a", "consent": True, "license": "owned"},
    {"id": "tts-002", "text": "Second recording from the same session.", "audio": "tts/session-a/002.wav", "speaker_id": "speaker-1", "session_id": "session-a", "consent": True, "license": "owned"},
    {"id": "tts-003", "text": "A disjoint validation session.", "audio": "tts/session-b/001.wav", "speaker_id": "speaker-1", "session_id": "session-b", "consent": True, "license": "owned"},
    {"id": "tts-004", "text": "Keep provenance beside each transcript.", "audio": "tts/session-b/002.wav", "speaker_id": "speaker-1", "session_id": "session-b", "consent": True, "license": "owned"},
]

asr_records = [
    {"id": "asr-001", "audio": "asr/a/001.wav", "text": "First verified transcript.", "speaker_id": "speaker-a"},
    {"id": "asr-002", "audio": "asr/a/002.wav", "text": "Second verified transcript.", "speaker_id": "speaker-a"},
    {"id": "asr-003", "audio": "asr/b/001.wav", "text": "Held out speaker transcript.", "speaker_id": "speaker-b"},
    {"id": "asr-004", "audio": "asr/b/002.wav", "text": "Another held out utterance.", "speaker_id": "speaker-b"},
]

vad_records = [
    {"id": "vad-001", "audio": "vad/001.wav", "segments": [{"start": 0.5, "end": 2.0, "label": "speech"}]},
    {"id": "vad-002", "audio": "vad/002.wav", "segments": [{"start": 1.0, "end": 4.0, "label": "speech"}]},
]

## 3. Validate and split

Portable datasets validate fields without decoding audio. Set `validate_files=True` only after the files exist.

In [ ]:
from voicehub.training import ASRDataset, SpeechDataset, TTSDataset

tts_source = TTSDataset(tts_records, model_type=TTS_MODEL_TYPE, root=DATA_ROOT, validate_files=False)
asr_source = ASRDataset(asr_records, model_type=ASR_MODEL_TYPE, root=DATA_ROOT, validate_files=False)
vad_source = SpeechDataset(vad_records, required_fields=("audio", "segments"))

tts_train, tts_validation = tts_source.train_test_split(
    validation_fraction=0.5, seed=42, group_by="session_id"
)
asr_train, asr_validation = asr_source.train_test_split(
    validation_fraction=0.5, seed=42, group_by="speaker_id"
)

tts_train_groups = {row["session_id"] for row in tts_train}
tts_validation_groups = {row["session_id"] for row in tts_validation}
asr_train_groups = {row["speaker_id"] for row in asr_train}
asr_validation_groups = {row["speaker_id"] for row in asr_validation}
assert tts_train_groups.isdisjoint(tts_validation_groups)
assert asr_train_groups.isdisjoint(asr_validation_groups)
print({"tts": (len(tts_train), len(tts_validation)), "asr": (len(asr_train), len(asr_validation)), "vad": len(vad_source)})

## 4. Inspect the selected contracts

In [ ]:
from voicehub.training import get_asr_dataset_spec, get_tts_dataset_spec

tts_contract = get_tts_dataset_spec(TTS_MODEL_TYPE)
asr_contract = get_asr_dataset_spec(ASR_MODEL_TYPE)
print(tts_contract.model_type, tts_contract.architecture.value, tts_contract.readiness.value)
print(asr_contract.model_type, asr_contract.architecture.value, asr_contract.readiness.value)

## 5. Write immutable split manifests

In [ ]:
if WRITE_MANIFESTS:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    tts_train.to_jsonl(OUTPUT_ROOT / "tts-train.jsonl")
    tts_validation.to_jsonl(OUTPUT_ROOT / "tts-validation.jsonl")
    asr_train.to_jsonl(OUTPUT_ROOT / "asr-train.jsonl")
    asr_validation.to_jsonl(OUTPUT_ROOT / "asr-validation.jsonl")
    print(OUTPUT_ROOT)

## 6. Validate audio and prepare model inputs

Run these cells only after real files are available.

In [ ]:
if RUN_AUDIO_VALIDATION:
    from voicehub import load_audio

    audio_path = Path(tts_source[0]["audio"])
    if not audio_path.is_file():
        raise FileNotFoundError(audio_path)
    audio = load_audio(audio_path, target_sampling_rate=44_100)
    print(audio.waveform.shape, audio.sampling_rate, audio.duration)

In [ ]:
if RUN_MODEL_PREPARATION:
    from voicehub import AutoModelForTextToSpeech

    training_model = AutoModelForTextToSpeech.from_pretrained(
        "nari-labs/Dia-1.6B-0626",
        model_type=TTS_MODEL_TYPE,
        device="cuda",
    )
    prepared = training_model.create_training_dataset(tts_train)
    batch = prepared.collate_fn([prepared[0]])
    print({name: getattr(value, "shape", type(value).__name__) for name, value in batch.items()})

## Next

Read the [data guide](https://kadirnar.github.io/voicehub/guides/data-preparation/), [ASR/VAD data guide](https://kadirnar.github.io/voicehub/guides/speech-data/), and [training matrix](https://kadirnar.github.io/voicehub/models/training-support/), then continue with [training.ipynb](training.ipynb).